# Training data collection, preprocessing, training, and validation

This notebook only sets variables, calls `.py` functions, and displays tables/plots.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'acquisition.py').exists():
    ROOT = Path(r'D:/BME/BCI/online_bci/online_eeg')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from acquisition import AcquisitionConfig, collect_training_segments
from preprocessing import AudioLabelConfig, PreprocessConfig, preprocess_many_recordings
from training import TrainingConfig, WindowConfig, train_validate_pipeline
from plots import plot_labeled_recording

print('Pipeline root:', ROOT)


## Settings

In [ ]:
RUN_ID = 'run_001'
RUN_DIR = ROOT / 'runs' / RUN_ID
RAW_DIR = RUN_DIR / 'raw_training'
LABELED_DIR = RUN_DIR / 'labeled_training'

COLLECT_TRAINING = False
TRAIN_SEGMENT_SEC = 300.0
TRAIN_SEGMENT_NAMES = ('train_01',)
TRAIN_RAW_NPZ = []

EEG_CHANNELS = (1, 2, 3, 4)
EEG_CHANNEL_NAMES = ('O1', 'Oz', 'O2', 'POz')
AUDIO_CHANNEL = 16
APPLY_SOFTWARE_FILTERS = False  # BIOPAC hardware already bandpasses the EEG at 1-35 Hz.
SOFTWARE_BANDPASS_HZ = None  # Set to (1.0, 35.0) only if APPLY_SOFTWARE_FILTERS=True.
SOFTWARE_NOTCH_HZ = None  # Set to (60.0,) only if APPLY_SOFTWARE_FILTERS=True.
SLIDING_WINDOW_SEC = 2.0
SLIDING_STRIDE_SEC = 0.05
LABEL_MODE = 'endpoint'  # use 'majority' to label each window by the most common sample label.

ACQ = AcquisitionConfig(
    samplerate=200,
    channels=(*EEG_CHANNELS, AUDIO_CHANNEL),
    chunk_sec=TRAIN_SEGMENT_SEC,
)

PRE = PreprocessConfig(
    eeg_channels=EEG_CHANNELS,
    audio_channel=AUDIO_CHANNEL,
    apply_software_filters=APPLY_SOFTWARE_FILTERS,
    bandpass_low_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[0],
    bandpass_high_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[1],
    notch_hz=SOFTWARE_NOTCH_HZ,
    notch_quality_factor=30.0,
    filter_order=4,
    demean_channels=True,
)

LABELS = AudioLabelConfig(
    class_names=('Eyes Open', 'Eyes Closed'),
    baseline_label=0,
    active_label=1,
    cue_label_sequence=None,
    alternate_binary_labels=True,
    label_duration_sec=None,  # transition mode: each cue switches state until the next cue.
    label_start_offset_sec=0.0,  # label switch starts exactly at cue onset.
    envelope_window_sec=0.025,
    onset_threshold=None,
    onset_min_interval_sec=0.50,
)

WIN = WindowConfig(
    feature_mode='filtered_signal',
    window_sec=SLIDING_WINDOW_SEC,
    stride_sec=SLIDING_STRIDE_SEC,
    label_mode=LABEL_MODE,
)

TRAIN = TrainingConfig(
    train_fraction=1.0,
    hidden_size=64,
    num_layers=2,
    dropout=0.2,
    batch_size=64,
    epochs=20,
    lr=1e-3,
    seed=888,
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
LABELED_DIR.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)


## Collect 5-minute training segments

In [ ]:
if COLLECT_TRAINING:
    train_raw_paths = collect_training_segments(
        output_dir=RAW_DIR,
        segment_names=TRAIN_SEGMENT_NAMES,
        config=ACQ,
        duration_sec=TRAIN_SEGMENT_SEC,
    )
else:
    train_raw_paths = [Path(p) for p in TRAIN_RAW_NPZ]

if not train_raw_paths:
    raise ValueError('Set COLLECT_TRAINING=True or populate TRAIN_RAW_NPZ with existing raw .npz files.')

train_raw_paths


## Preprocess and audio-channel labels

In [ ]:
labeled_paths = preprocess_many_recordings(
    raw_npz_paths=train_raw_paths,
    output_dir=LABELED_DIR,
    preprocess_config=PRE,
    label_config=LABELS,
)

labeled_paths


In [ ]:
fig, axes = plot_labeled_recording(
    labeled_paths[0],
    max_duration_sec=None,
    channel_names=EEG_CHANNEL_NAMES,
)


## Train and validate LSTM

In [ ]:
train_result = train_validate_pipeline(
    labeled_npz_paths=labeled_paths,
    output_dir=RUN_DIR,
    window_config=WIN,
    training_config=TRAIN,
)

checkpoint_path = train_result['checkpoint_path']
print('Saved model checkpoint:', checkpoint_path)
print('Aligned validation EEG/prediction CSV:', train_result['validation_aligned_prediction_csv'])
checkpoint_path


In [ ]:
history = train_result['history']
display(history.tail())
display(train_result['validation_summary'])
display(train_result['validation_per_class'])

ax = history.plot(x='epoch', y=['train_acc', 'val_acc'], marker='o', figsize=(8, 4))
ax.set_ylabel('Accuracy')
ax.set_title('Training and validation accuracy')
ax.grid(True, alpha=0.3)
plt.tight_layout()
